# StreamLens - Exploratory Data Analysis (EDA)
## MovieLens Dataset Analysis

**Objective:** Understand the structure, quality, and characteristics of the MovieLens dataset

**Dataset:** MovieLens Latest Small
- ~100,000 ratings
- ~9,000 movies
- Applied by ~600 users

**Analysis Steps:**
1. Load and inspect datasets
2. Data quality check
3. Statistical analysis
4. Visualizations
5. Insights for recommendation system

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## 1. Load Datasets

In [ ]:
# Define data paths
DATA_PATH = '../data/raw/ml-latest-small/'

# Load datasets
movies = pd.read_csv(DATA_PATH + 'movies.csv')
ratings = pd.read_csv(DATA_PATH + 'ratings.csv')
tags = pd.read_csv(DATA_PATH + 'tags.csv')
links = pd.read_csv(DATA_PATH + 'links.csv')

print(f"✅ Loaded {len(movies)} movies")
print(f"✅ Loaded {len(ratings)} ratings")
print(f"✅ Loaded {len(tags)} tags")
print(f"✅ Loaded {len(links)} links")

## 2. Dataset Overview

In [ ]:
# Movies dataset
print("=" * 50)
print("MOVIES DATASET")
print("=" * 50)
print(f"Shape: {movies.shape}")
print(f"\nColumns: {list(movies.columns)}")
print(f"\nFirst 5 rows:")
display(movies.head())
print(f"\nData types:")
print(movies.dtypes)
print(f"\nMissing values:")
print(movies.isnull().sum())

In [ ]:
# Ratings dataset
print("=" * 50)
print("RATINGS DATASET")
print("=" * 50)
print(f"Shape: {ratings.shape}")
print(f"\nColumns: {list(ratings.columns)}")
print(f"\nFirst 5 rows:")
display(ratings.head())
print(f"\nData types:")
print(ratings.dtypes)
print(f"\nMissing values:")
print(ratings.isnull().sum())

# Convert timestamp to datetime
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
print(f"\nRating date range: {ratings['datetime'].min()} to {ratings['datetime'].max()}")

In [ ]:
# Tags dataset
print("=" * 50)
print("TAGS DATASET")
print("=" * 50)
print(f"Shape: {tags.shape}")
print(f"\nFirst 10 rows:")
display(tags.head(10))
print(f"\nMissing values:")
print(tags.isnull().sum())

## 3. Statistical Analysis

In [ ]:
# Basic statistics
print("=" * 50)
print("KEY STATISTICS")
print("=" * 50)

num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()
num_ratings = len(ratings)
avg_rating = ratings['rating'].mean()
median_rating = ratings['rating'].median()

print(f"Total Users: {num_users:,}")
print(f"Total Movies: {num_movies:,}")
print(f"Total Ratings: {num_ratings:,}")
print(f"Average Rating: {avg_rating:.2f}")
print(f"Median Rating: {median_rating:.2f}")
print(f"Sparsity: {(1 - num_ratings / (num_users * num_movies)) * 100:.2f}%")
print(f"\nRatings per user: {num_ratings / num_users:.2f}")
print(f"Ratings per movie: {num_ratings / num_movies:.2f}")

In [ ]:
# Rating distribution
print("\nRating Distribution:")
print(ratings['rating'].value_counts().sort_index())

# Statistical summary
print("\nRating Statistics:")
print(ratings['rating'].describe())

## 4. Data Visualizations

In [ ]:
# Rating distribution histogram
plt.figure(figsize=(10, 6))
sns.countplot(data=ratings, x='rating', palette='viridis')
plt.title('Distribution of Movie Ratings', fontsize=16, fontweight='bold')
plt.xlabel('Rating', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
for i, v in enumerate(ratings['rating'].value_counts().sort_index().values):
    plt.text(i, v + 1000, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Ratings over time
ratings['year'] = ratings['datetime'].dt.year
ratings['month'] = ratings['datetime'].dt.to_period('M')

ratings_over_time = ratings.groupby('year').size()

plt.figure(figsize=(12, 6))
ratings_over_time.plot(kind='bar', color='steelblue')
plt.title('Number of Ratings per Year', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Ratings', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 most rated movies
movie_rating_counts = ratings.groupby('movieId').size().reset_index(name='num_ratings')
top_movies = movie_rating_counts.nlargest(20, 'num_ratings')
top_movies = top_movies.merge(movies[['movieId', 'title']], on='movieId')

plt.figure(figsize=(12, 8))
sns.barplot(data=top_movies, y='title', x='num_ratings', palette='rocket')
plt.title('Top 20 Most Rated Movies', fontsize=16, fontweight='bold')
plt.xlabel('Number of Ratings', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.tight_layout()
plt.show()

print("\nTop 20 Most Rated Movies:")
display(top_movies)

In [ ]:
# User activity distribution
user_activity = ratings.groupby('userId').size()

plt.figure(figsize=(12, 6))
plt.hist(user_activity.values, bins=50, color='coral', edgecolor='black')
plt.title('Distribution of Ratings per User', fontsize=16, fontweight='bold')
plt.xlabel('Number of Ratings', fontsize=12)
plt.ylabel('Number of Users', fontsize=12)
plt.axvline(user_activity.median(), color='red', linestyle='--', label=f'Median: {user_activity.median():.0f}')
plt.axvline(user_activity.mean(), color='blue', linestyle='--', label=f'Mean: {user_activity.mean():.0f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nUser Activity Stats:")
print(f"Mean ratings per user: {user_activity.mean():.2f}")
print(f"Median ratings per user: {user_activity.median():.2f}")
print(f"Max ratings by a user: {user_activity.max()}")
print(f"Min ratings by a user: {user_activity.min()}")

## 5. Genre Analysis

In [ ]:
# Extract and count genres
all_genres = []
for genres_str in movies['genres']:
    if pd.notna(genres_str):
        all_genres.extend(genres_str.split('|'))

genre_counts = pd.Series(all_genres).value_counts()

plt.figure(figsize=(12, 8))
genre_counts.head(20).plot(kind='barh', color='teal')
plt.title('Top 20 Movie Genres', fontsize=16, fontweight='bold')
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nGenre Distribution:")
print(genre_counts)

In [ ]:
# Average rating by genre
genre_ratings = []

for genre in genre_counts.index[:15]:  # Top 15 genres
    genre_movies = movies[movies['genres'].str.contains(genre, na=False)]['movieId']
    genre_rating_avg = ratings[ratings['movieId'].isin(genre_movies)]['rating'].mean()
    genre_ratings.append({'genre': genre, 'avg_rating': genre_rating_avg})

genre_ratings_df = pd.DataFrame(genre_ratings).sort_values('avg_rating', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=genre_ratings_df, x='avg_rating', y='genre', palette='coolwarm')
plt.title('Average Rating by Genre (Top 15 Genres)', fontsize=16, fontweight='bold')
plt.xlabel('Average Rating', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.xlim(3.0, 4.0)
plt.tight_layout()
plt.show()

print("\nAverage Rating by Genre:")
display(genre_ratings_df)

## 6. Data Quality Assessment

In [ ]:
print("=" * 50)
print("DATA QUALITY ASSESSMENT")
print("=" * 50)

# Check for duplicates
print("\n1. Duplicate Check:")
print(f"   Movies duplicates: {movies.duplicated().sum()}")
print(f"   Ratings duplicates: {ratings.duplicated().sum()}")
print(f"   Tags duplicates: {tags.duplicated().sum()}")

# Check for missing values
print("\n2. Missing Values:")
print("   Movies:")
print(movies.isnull().sum())
print("\n   Ratings:")
print(ratings.isnull().sum())

# Check rating range
print("\n3. Rating Range:")
print(f"   Min rating: {ratings['rating'].min()}")
print(f"   Max rating: {ratings['rating'].max()}")
print(f"   Unique ratings: {sorted(ratings['rating'].unique())}")

# Movies without ratings
movies_with_ratings = ratings['movieId'].unique()
movies_without_ratings = set(movies['movieId']) - set(movies_with_ratings)
print(f"\n4. Movies without ratings: {len(movies_without_ratings)}")

# Cold start analysis
print("\n5. Cold Start Analysis:")
movies_with_few_ratings = movie_rating_counts[movie_rating_counts['num_ratings'] < 5]
users_with_few_ratings = user_activity[user_activity < 20]
print(f"   Movies with < 5 ratings: {len(movies_with_few_ratings)} ({len(movies_with_few_ratings)/len(movies)*100:.1f}%)")
print(f"   Users with < 20 ratings: {len(users_with_few_ratings)} ({len(users_with_few_ratings)/num_users*100:.1f}%)")

## 7. Key Insights & Recommendations

### Findings:
1. **Dataset Size**: The dataset contains sufficient data for building a recommendation system
2. **Data Quality**: Minimal missing values, clean dataset
3. **Sparsity**: High sparsity is typical for recommendation systems
4. **Rating Bias**: Users tend to give higher ratings (positive skew)
5. **Cold Start**: Significant number of movies and users with few interactions

### Implications for Recommendation System:
1. **Hybrid Approach**: Use both content-based and collaborative filtering to handle cold start
2. **Genre Features**: Strong signal for content-based recommendations
3. **Popular Items**: Use popularity as a baseline for new users
4. **Temporal Context**: Consider time-based patterns in recommendations
5. **Context-Aware**: Genre preferences can serve as mood indicators

### Next Steps:
1. Feature engineering (genre encoding, TF-IDF for tags)
2. Build baseline content-based recommender
3. Implement collaborative filtering
4. Add context-aware layer

In [ ]:
# Save processed data for later use
print("Saving insights summary...")

summary = {
    'num_users': num_users,
    'num_movies': num_movies,
    'num_ratings': num_ratings,
    'avg_rating': avg_rating,
    'sparsity': (1 - num_ratings / (num_users * num_movies)) * 100,
    'top_genres': genre_counts.head(10).to_dict(),
    'date_range': f"{ratings['datetime'].min()} to {ratings['datetime'].max()}"
}

import json
with open('../data/processed/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print("✅ Summary saved to data/processed/eda_summary.json")
print("\n✅ EDA Complete!")